## Current research direction — HRI adaptation

Reusable preparation: review and adapt these steps for endemic-bird HRI assessment. Any legacy forest-loss target analysis in this notebook is exploratory, not an HRI definition.

The original code and saved outputs are retained. No cells were executed during restoration. Use `docs/methodology/hri_workflow.md` for the current direction and `notebooks/hri/` for the simple lecturer-facing versions. Do not describe existing forest-loss outputs as habitat-risk classes.

# 03 — Source verification and evidence log

**Purpose:** document what the project files demonstrate, distinguish reported sources from verified provenance, and record what must be recovered before modelling.

This notebook does not verify a provider merely because its name appears in an earlier notebook. No original provider export, raster metadata, or extraction script has been established by the current evidence. Unknown details remain blank in the editable source registry.

Run from the project root or `notebooks/` using the project `.venv` kernel. Raw data and existing review tables are read only. This notebook initializes a separate registry only if it does not exist, preserves subsequent edits, and refreshes derived audit reports.

In [1]:
from pathlib import Path
import hashlib
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "requirements-research-lock.txt").is_file()
     and (p / "data/raw/observations/final_dataset_complete.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run inside the component4 project.")

RAW_PATH = PROJECT_ROOT / "data/raw/observations/final_dataset_complete.csv"
METADATA = PROJECT_ROOT / "data/metadata"
AUDITS = PROJECT_ROOT / "reports/audits"
METADATA.mkdir(parents=True, exist_ok=True)
AUDITS.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(RAW_PATH)
notes = pd.read_csv(METADATA / "source_verification_notes.csv", keep_default_na=False)
print("Project:", PROJECT_ROOT)
print("Raw dataset shape:", raw.shape)
display(notes)

Project: /Users/keerthikeerthi/Desktop/component4
Raw dataset shape: (102929, 23)


,data_group,reported_source,pending_verification
0,Bird observations,eBird,Original export details and checklist IDs
1,Forest,Hansen GFC 2024 v1.12,"QGIS extraction method, masking and loss-year ..."
2,Weather,Kaggle; exact dataset unknown,"Dataset link, original provider, units, time i..."
3,Elevation,Kaggle; exact dataset unknown,"Dataset link, units, resolution and extraction..."


## 1. Evidence hierarchy

| Evidence | What it establishes | What it does not establish |
| --- | --- | --- |
| Raw combined CSV | Actual stored values, columns and coverage | Original provider, units, licence or correct joins |
| Source-verification notes | Sources previously reported by the project | Independent confirmation of those sources |
| Cleaning notebook and review CSVs | Current transformations, flags and table relationships | Upstream extraction correctness |
| Original exports, metadata and extraction scripts — still to recover | Provider release, units, masks and reproducible extraction | Ecological validity without additional assessment |

The original file name contains “final”, but the dataset is still under review. File checksums below identify the exact local evidence used; they do not certify scientific correctness.

In [2]:
evidence_paths = [RAW_PATH, METADATA / "source_verification_notes.csv"]
evidence_paths += sorted((PROJECT_ROOT / "data/interim").glob("*.csv"))
evidence_paths += [PROJECT_ROOT / "notebooks/01_data_understanding.ipynb",
                   PROJECT_ROOT / "notebooks/02_data_cleaning.ipynb"]
inventory = pd.DataFrame([
    {"path": str(p.relative_to(PROJECT_ROOT)), "bytes": p.stat().st_size,
     "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in evidence_paths
])
display(inventory)
print("Files currently present under data/raw/:")
for p in sorted((PROJECT_ROOT / "data/raw").rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJECT_ROOT))

,path,bytes,sha256
0,data/raw/observations/final_dataset_complete.csv,15397822,90ecf906d38c6586a39348c0471ed5a75e3f2497f55c54...
1,data/metadata/source_verification_notes.csv,413,368a615bf7c0892c8dd4c77120524aa1aef89bf6c49c7c...
2,data/interim/dataset_review_stage.csv,25205534,b13881a4e7e2f5c50b5838f4d7166db08c6ec7ea855cf6...
3,data/interim/eda_outlier_limits.csv,226,a707e6a9d52d4e4b634993138ae83dc1e3b84e0e4fc0e6...
4,data/interim/environment_location_date_review.csv,2579801,74d7043762d4e246b3ea9c398b9457e703f14eab100a65...
5,data/interim/location_date_weather_review.csv,1446052,d0b83dee979e5f1512d9fb07ace15cb918b14923e16d04...
6,data/interim/location_forest_review.csv,128829,8a457f059e203f82b58753662f3272e7f739975bb210f4...
7,data/interim/location_reference.csv,225629,07ea9cc916bab5071c1f5d680425f50dc5ed805a77276c...
8,data/interim/review_flag_summary.csv,240,f51e7a172fa499f3b2eb065b01aea559b1f130cfc1268b...
9,data/interim/species_location_review.csv,1951043,e1fbe050620d8e9d00fedac15c17f95c3b71f2c35a24f2...


Files currently present under data/raw/:
data/raw/observations/final_dataset_complete.csv


## 2. Editable source registry

The registry is `data/metadata/source_registry.csv`. Edit that CSV when you recover evidence, then rerun this notebook. It is intentionally separate from the earlier notes because Notebook 02 regenerates those notes from its inline table.

Record exact provider URLs and local evidence paths, rather than a generic search page or the word “Kaggle”. A discovery link alone does not confirm that the source produced this CSV. `verification_status` is a researcher-entered status, not an automatically certified result. Use `unverified`, `partially_verified`, or `verified`; explain the evidence in `verification_notes`.

Do not enter guessed units, versions, download dates, checklist IDs or extraction methods.

In [3]:
registry_path = METADATA / "source_registry.csv"
if not registry_path.exists():
    registry = notes.rename(columns={"pending_verification": "remaining_questions"}).copy()
    for column in ["source_url", "original_filename", "source_version", "download_date",
                   "units", "crs", "spatial_resolution", "geographic_coverage",
                   "temporal_coverage", "no_data_definition", "licence",
                   "extraction_or_join_method", "local_evidence_paths",
                   "verification_notes", "reviewed_by", "reviewed_on"]:
        registry[column] = ""
    registry["verification_status"] = "unverified"
    registry.to_csv(registry_path, index=False)
    print("Initialized registry with reported sources and unknown details.")
else:
    print("Using existing registry; manual entries were preserved.")
registry = pd.read_csv(registry_path, keep_default_na=False)
display(registry)

Using existing registry; manual entries were preserved.


,data_group,reported_source,remaining_questions,source_url,original_filename,source_version,download_date,units,crs,spatial_resolution,geographic_coverage,temporal_coverage,no_data_definition,licence,extraction_or_join_method,local_evidence_paths,verification_notes,reviewed_by,reviewed_on,verification_status
0,Bird observations,eBird,Original export details and checklist IDs,,,,,,,,,,,,,,,,,unverified
1,Forest,Hansen GFC 2024 v1.12,"QGIS extraction method, masking and loss-year ...",,,,,,,,,,,,,,,,,unverified
2,Weather,Kaggle; exact dataset unknown,"Dataset link, original provider, units, time i...",,,,,,,,,,,,,,,,,unverified
3,Elevation,Kaggle; exact dataset unknown,"Dataset link, units, resolution and extraction...",,,,,,,,,,,,,,,,,unverified


## 3. Reproducible checks on the current combined CSV

These are verified local observations, not conclusions about why a problem occurred. Elevation repetition, for example, is a reason to inspect the upstream join; it is not proof that the values are wrong. Duplicate rows cannot be resolved reliably without understanding the original recording identifiers.

In [4]:
dates = pd.to_datetime(raw["observation_date"], format="%Y-%m-%d", errors="coerce")
counts = pd.to_numeric(raw["observation_count"], errors="coerce")
checks = pd.DataFrame([
    ("Observation rows", len(raw)),
    ("Columns", raw.shape[1]),
    ("Distinct scientific names", raw["scientific_name"].nunique()),
    ("Exact coordinate pairs", len(raw[["latitude", "longitude"]].drop_duplicates())),
    ("Earliest observation", str(dates.min())),
    ("Latest observation", str(dates.max())),
    ("Invalid observation dates", int(dates.isna().sum())),
    ("Extra exact duplicate rows", int(raw.duplicated().sum())),
    ("Rows in duplicate groups including first copies", int(raw.duplicated(keep=False).sum())),
    ("Missing duration", int(raw["duration_minutes"].isna().sum())),
    ("Missing distance", int(raw["effort_distance_km"].isna().sum())),
    ("Nonnumeric or missing observation counts", int(counts.isna().sum())),
    ("Duration greater than 1440 minutes", int(raw["duration_minutes"].gt(1440).sum())),
    ("Distinct elevation values", raw["elevation"].nunique()),
    ("Rain equals precipitation on all rows", bool(raw["rain_sum"].eq(raw["precipitation_sum"]).all())),
    ("Loss after observation year", int((raw["forest_loss_year"].gt(0) & raw["forest_loss_year"].gt(dates.dt.year)).sum())),
    ("Loss in observation year", int(raw["forest_loss_year"].eq(dates.dt.year).sum())),
], columns=["check", "observed_value"])
display(checks)
print("Unparsed count values:")
display(raw.loc[counts.isna(), "observation_count"].value_counts(dropna=False))
print("Locations per recorded elevation:")
display(raw[["latitude", "longitude", "elevation"]].drop_duplicates()
        .groupby("elevation").size().rename("coordinate_pairs"))

,check,observed_value
0,Observation rows,102929
1,Columns,23
2,Distinct scientific names,25
3,Exact coordinate pairs,7245
4,Earliest observation,2010-01-01 00:00:00
5,Latest observation,2023-06-17 00:00:00
6,Invalid observation dates,0
7,Extra exact duplicate rows,18411
8,Rows in duplicate groups including first copies,31667
9,Missing duration,11401


Unparsed count values:


observation_count
X    12743
Name: count, dtype: int64

Locations per recorded elevation:


elevation
0.0        424
5.0        723
6.0        246
7.0         86
8.0         72
10.0       420
12.0       760
15.0       416
16.0       164
18.0        99
20.0       201
27.0       764
124.0      126
125.0      114
376.0      643
510.0      534
652.0      403
1281.0    1050
Name: coordinate_pairs, dtype: int64

## 4. Source-specific evidence and recovery tasks

### Bird observations

**Reported source:** eBird, according to the existing project notes. The combined file contains species, dates, coordinates, count and effort fields, but no checklist identifier column.

**Recover:** original export and export description, checklist identifiers, meaning of `X`, protocol definitions and filtering history. Establish whether repeated rows represent repeated imports, shared lists or separate events whose identifiers were discarded. Do not interpret missing species records as absence or convert `X` to zero without source support.

### Weather

**Reported source:** Kaggle; exact dataset unknown. Rain and precipitation are identical in this file. Bird observation dates do not establish that weather values were measured at those dates.

**Recover:** exact dataset and upstream provider, units, temporal aggregation, station/grid locations and joining code. Determine whether matching used coordinates, city, nearest station or another rule. Observation-day weather cannot automatically be averaged into a representative annual climate series.

### Elevation

**Reported source:** Kaggle; exact dataset unknown. The audit finds only 18 distinct values across thousands of coordinates.

**Recover:** source DEM or table, vertical units, spatial resolution and join/extraction method. Inspect a few known locations against the original source. Record whether the values are point elevations or broad location summaries before retaining or replacing them.

### Forest

**Reported source:** Hansen GFC 2024 v1.12, according to project notes. Original raster evidence and the QGIS extraction method have not been established in this workspace.

**Recover:** exact release and raster bands, no-data/validity masks, CRS, extraction geometry and loss-year conversion. Confirm whether values are pixel samples, buffer summaries or area fractions. Do not add 2000 to a value already converted to a calendar year. Distinguish no mapped loss from missing/unobserved data.

Treat `treecover2000` as historical context, not measured current canopy. The meaning must ultimately be checked against the actual source release used. Forest loss later than an observation is useful for auditing or constructing labels, but cannot be used as a predictor available at that observation date.

## 5. Recheck the existing next-year target experiment

This reproduces the current notebook's restriction to locations observed in consecutive years. It is an audit of that candidate design, not an approved target. Loss-code interpretation and observation coverage still require provenance verification.

If there are zero positive labels, that subset cannot support a two-class loss classifier. It does not demonstrate an absence of forest loss across the study area. Investigate area-based cell-year outcomes from verified rasters; an environmental label need not require a bird visit in the later year.

In [5]:
keys = ["latitude", "longitude"]
forest_columns = ["forest_loss", "forest_loss_year"]
variation = raw.groupby(keys)[forest_columns].nunique(dropna=False)
if variation.gt(1).any().any():
    raise ValueError("Conflicting forest values by location; resolve before target audit.")
forest = raw[keys + forest_columns].drop_duplicates()
location_years = raw[keys].assign(year=dates.dt.year).dropna(subset=["year"]).drop_duplicates()
following = location_years.rename(columns={"year": "next_year"})
pairs = location_years.assign(next_year=lambda d: d["year"] + 1).merge(
    following, on=keys + ["next_year"], validate="one_to_one")
pairs = pairs.merge(forest, on=keys, validate="many_to_one")
previous_loss = pairs["forest_loss_year"].gt(0) & pairs["forest_loss_year"].le(pairs["year"])
eligible = pairs.loc[~previous_loss].copy()
positive = eligible["forest_loss_year"].eq(eligible["next_year"])
target_audit = pd.DataFrame([
    ("Consecutive location-year pairs", len(pairs)),
    ("Pairs without loss recorded by starting year", len(eligible)),
    ("Recorded next-year positive pairs", int(positive.sum())),
], columns=["check", "observed_value"])
display(target_audit)
if positive.sum() == 0:
    print("Current candidate subset has no positive next-year examples; do not train a binary classifier on it.")

,check,observed_value
0,Consecutive location-year pairs,1661
1,Pairs without loss recorded by starting year,1638
2,Recorded next-year positive pairs,0


Current candidate subset has no positive next-year examples; do not train a binary classifier on it.


## 6. Verification checklist and next action

Start by recovering the weather and elevation source links and join method. Add evidence paths and findings to the registry. Then resolve the bird export/checklist questions and forest extraction details. Units, spatial support and timing must be known before deriving risk indicators.

A source is ready for scientific use only after its identity, relevant units/coverage and extraction have been supported by evidence. A filled-in registry alone is insufficient. Review several joined records against original data and document discrepancies.

Before modelling: confirm the study area/species, resolve cleaning decisions, establish spatial analysis units, verify complete later-label windows, and count events in every evaluation partition. No HRI weights, validated target or model results are created by this notebook.

In [6]:
required_evidence = ["source_url", "original_filename", "units", "temporal_coverage",
                     "extraction_or_join_method", "local_evidence_paths"]
completeness = registry[["data_group", "verification_status"]].copy()
completeness["unfilled_core_fields"] = registry.apply(
    lambda row: "; ".join(field for field in required_evidence
                          if not str(row.get(field, "")).strip()), axis=1)
display(completeness)
print("This checks documentation completeness only; it does not verify source authenticity.")

,data_group,verification_status,unfilled_core_fields
0,Bird observations,unverified,source_url; original_filename; units; temporal...
1,Forest,unverified,source_url; original_filename; units; temporal...
2,Weather,unverified,source_url; original_filename; units; temporal...
3,Elevation,unverified,source_url; original_filename; units; temporal...


This checks documentation completeness only; it does not verify source authenticity.


## 7. Save the audit evidence

The following cell refreshes derived reports under `reports/audits/`. The editable registry is not overwritten. Existing raw/review data are not changed. Rerun after updating sources so the evidence snapshot describes the current files.

In [7]:
inventory.to_csv(AUDITS / "source_evidence_inventory.csv", index=False)
checks.to_csv(AUDITS / "source_verification_checks.csv", index=False)
target_audit.to_csv(AUDITS / "source_target_feasibility.csv", index=False)
completeness.to_csv(AUDITS / "source_registry_completeness.csv", index=False)
summary = [
    "# Source verification audit",
    "", "Generated (UTC): " + datetime.now(timezone.utc).isoformat(),
    "", "Source identities remain subject to the evidence recorded in data/metadata/source_registry.csv.",
    "Documentation completeness is not independent source verification.",
    "", "## Current file checks", "", checks.to_string(index=False),
    "", "## Candidate target audit", "", target_audit.to_string(index=False),
    "", "## Next action", "",
    "Recover the exact weather/elevation sources and upstream join method, then record supporting evidence in the registry.",
]
(AUDITS / "source_verification_summary.md").write_text("\n".join(summary) + "\n")
print("Saved evidence inventory, checks, target audit, completeness report and summary to:", AUDITS)
print("Editable source registry:", registry_path)

Saved evidence inventory, checks, target audit, completeness report and summary to: /Users/keerthikeerthi/Desktop/component4/reports/audits
Editable source registry: /Users/keerthikeerthi/Desktop/component4/data/metadata/source_registry.csv


# HRI source checkpoint

These reports document downloaded inputs and known limitations. The original next-year target feasibility section is historical and is not HRI validation. Species assessments are source snapshots; complete trait evidence and temporal interpretation before scoring. Fragmentation, human pressure, HRI weights and risk classes are not yet verified inputs.

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd()
while not (ROOT / "configs/study.json").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run within component4")
    ROOT = ROOT.parent
for name in ["forest_input_audit.json", "weather_source_validation.json", "terrain_features.json"]:
    path = ROOT / "reports/audits" / name
    print(name)
    print(json.dumps(json.loads(path.read_text()), indent=2))
print("These are source checks, not validated HRI scores.")